# 02 — Retrieval e RAG fundamentado

Este notebook valida a recuperação antes da geração, e depois liga as duas. Separar as etapas permite descobrir se uma resposta ruim nasceu na busca ou no modelo.

In [ ]:
import sys
import textwrap
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ragnaldo.config import GENERATION, SETTINGS
from ragnaldo.ingestion import load_verified_index


def wrap(texto, largura=88):
    """Quebra linhas longas preservando os paragrafos.

    O Jupyter nao quebra linha na saida por padrao, e um chunk de mil caracteres
    vira uma linha unica com barra de rolagem. Fazer isso aqui, e nao na
    configuracao do JupyterLab, garante que quem abrir o notebook depois veja o
    mesmo que voce.
    """
    return '\n'.join(
        textwrap.fill(linha, largura) if linha.strip() else linha
        for linha in str(texto).splitlines()
    )


vector_store, manifest = load_verified_index()
print(f"{manifest['chunk_count']} chunks | modelo gerador: {GENERATION.model}")

## 1. Busca por similaridade

Cada resultado traz a distância. Com vetores normalizados, o FAISS devolve a distância L2 ao quadrado: 0 é idêntico, 2 é ortogonal. Ou seja, menor é melhor — e o número importa, porque é ele que decide se há evidência suficiente para responder.

In [ ]:
question = 'O que o ONE AI for Tech ensina sobre RAG e LangChain?'
results = vector_store.similarity_search_with_score(question, k=SETTINGS.retrieval_k)

for rank, (document, score) in enumerate(results, start=1):
    print(f'[{rank}] distancia={score:.4f} | {document.metadata["source"]} | {document.metadata["location"]}')
    print(wrap(document.page_content[:300]), '\n')

## 2. O corte de evidência

A busca sempre devolve `k` resultados, mesmo para uma pergunta sobre algo que não existe no corpus: ela retorna os menos ruins, não os bons. Sem um corte, esses trechos irrelevantes chegam ao modelo como se fossem contexto legítimo, e aí só resta a ele improvisar.

`select_evidence` descarta o que está além de `max_distance`. O limiar atual (1.2, equivalente a similaridade de cosseno 0.4) é um chute honesto: barra o que está claramente fora do assunto, e nada além disso. A calibração real depende de dados, e é tarefa do notebook 04.

In [ ]:
from ragnaldo.generation import format_context, select_evidence

evidence = select_evidence(results)
print(f'{len(evidence)} de {len(results)} trechos passaram no corte (max_distance={GENERATION.max_distance})')

documents = [document for document, _ in evidence]
context = format_context(documents)
print(wrap(context[:800]))

## 3. O prompt

O rótulo de fonte já vem pronto no contexto. O modelo copia, não deduz — inventar referência é exatamente o que uma LLM faz com mais naturalidade, e a única defesa é nunca pedir que ela produza uma.

O humor fica subordinado à evidência e é proibido em recusas.

In [ ]:
from ragnaldo.generation import PROMPT

print(PROMPT.invoke({'context': '<contexto>', 'question': question}).to_string())

## 4. A cadeia completa

`answer_question` faz recuperação, corte, geração e registro numa chamada. Devolve a resposta, os documentos que a sustentam e o registro de execução exigido pelo card 8 do enunciado.

In [ ]:
from ragnaldo.generation import answer_question

answer, used, record = answer_question(question, vector_store)

print(wrap(answer))
print(f'\n--- {record.latency_ms} ms | {len(record.retrieved)} trechos | recusou: {record.refused} ---')
for chunk in record.retrieved:
    print(f'  {chunk.source} | {chunk.location} | distancia={chunk.distance}')

## 5. O teste que importa: a pergunta fora do corpus

Um RAG que responde bem ao que sabe é fácil. O que distingue um agente confiável é o que ele faz diante do que não sabe.

Aqui a recusa acontece antes da chamada à API: sem evidência, o modelo só produziria uma alucinação educada — e ainda cobraria por ela.

In [ ]:
fora = 'Qual foi o placar da final da Copa do Mundo de 1970?'
resposta_fora, docs_fora, registro_fora = answer_question(fora, vector_store)

print(wrap(resposta_fora))
print(f'\nrecusou: {registro_fora.refused} | trechos aceitos: {len(registro_fora.retrieved)} | {registro_fora.latency_ms} ms')

brutos = vector_store.similarity_search_with_score(fora, k=SETTINGS.retrieval_k)
print(f'\nA busca ainda devolveu {len(brutos)} resultados. Distancias: ' +
      ', '.join(f'{s:.3f}' for _, s in brutos))
print('Quanto passou do limiar virou contexto; o resto foi descartado.')

## 6. Registro de execução

Cada chamada vira uma linha em `artifacts/logs/execution.jsonl`: pergunta, resposta, trechos com distância, latência e timestamp. É a matéria-prima da calibração do limiar e do registro que o enunciado pede.

O arquivo fica fora do Git — contém as perguntas reais de quem usar o agente. Amostras curadas vão para o README.

In [ ]:
import json

from ragnaldo.config import EXECUTION_LOG_PATH

linhas = EXECUTION_LOG_PATH.read_text(encoding='utf-8').strip().splitlines()
print(f'{len(linhas)} execucoes registradas em {EXECUTION_LOG_PATH.relative_to(PROJECT_ROOT)}\n')

ultima = json.loads(linhas[-1])
print(json.dumps({k: v for k, v in ultima.items() if k != 'answer'}, ensure_ascii=False, indent=2))